### Imports

In [199]:
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import pyopencl as cl
import time

### 8a RGB-Bild einlesen und in ein Graustufenbild umwandeln + 
### 8b Helligkeit und Kontrast vom Graustufenbild anpassen

In [ ]:
image_path = Path("images_input/1.nature_small.jpeg")
## image_path = Path("images_input/2.nature_medium.jpeg")
## image_path = Path("images_input/3.nature_large.jpeg")
## image_path = Path("images_input/4.nature_mega.jpeg")

image = Image.open(image_path)
rgb_matrix = np.array(image, dtype=np.uint8)

height, width, channels = rgb_matrix.shape
pixel_count = height * width

rgb_flat = rgb_matrix.flatten()
gray_matrix_flat = np.empty(pixel_count, dtype=np.uint8)
histogram = np.zeros(256, dtype=np.int32)

alpha = np.float32(1.3)
beta = np.float32(20)

context = cl.create_some_context()
queue = cl.CommandQueue(context)

mf = cl.mem_flags

rgb_input_buffer = cl.Buffer(context, mf.READ_ONLY | mf.COPY_HOST_PTR, hostbuf=rgb_flat)
gray_output_buffer = cl.Buffer(context, mf.WRITE_ONLY, gray_matrix_flat.nbytes)
histogram_buffer = cl.Buffer(context, mf.READ_WRITE | mf.COPY_HOST_PTR, hostbuf=histogram)

program = cl.Program(context, """
#pragma OPENCL EXTENSION cl_khr_global_int32_base_atomics : enable
__kernel void rgb_to_grayscale_brightness_contrast(
    __global const uchar *rgb,
    __global uchar *gray_out,
    const float alpha,
    const float beta
)
{
    int pixel_id = get_global_id(0);
    int rgb_id = pixel_id * 3;

    uchar red = rgb[rgb_id];
    uchar green = rgb[rgb_id + 1];
    uchar blue = rgb[rgb_id + 2];

    uchar gray_value = (uchar)(0.21f * red + 0.72f * green + 0.07f * blue);
    float value = alpha * gray_value + beta;

    if (value < 0.0f) {
        value = 0.0f;
    }

    if (value > 255.0f) {
        value = 255.0f;
    }

    gray_out[pixel_id] = (uchar)(value);
}
__kernel void compute_histogram(
    __global const uchar *gray,
    __global int *histogram
)
{
    int pixel_id = get_global_id(0);
    uchar value = gray[pixel_id];
    atomic_inc(&histogram[value]);
}
""").build()


start = time.perf_counter()

program.rgb_to_grayscale_brightness_contrast(
    queue,
    (pixel_count,),
    None,
    rgb_input_buffer,
    gray_output_buffer,
    alpha,
    beta
)

program.compute_histogram(
    queue,
    (pixel_count,),
    None,
    gray_output_buffer,
    histogram_buffer
)

cl.enqueue_copy(queue, gray_matrix_flat, gray_output_buffer)
cl.enqueue_copy(queue, histogram, histogram_buffer)
queue.finish()

gray_matrix_2 = gray_matrix_flat.reshape((height, width))

end = time.perf_counter()

pyopencl_time = end - start
print(pyopencl_time)

## gray_matrix = gray_flat.reshape((height, width))
## plt.imshow(gray_matrix, cmap="gray", vmin=0, vmax=255)
## plt.axis("off")
## plt.show()

0.0027275001630187035


### 8c Histogramm der Grauwerte berechnen und darstellen

In [201]:
## histogram = np.bincount(gray_matrix_2.flatten(), minlength=256)

# plt.figure(figsize=(10, 5))
# plt.bar(range(256), histogram, width=1.0)
# plt.xlabel("Grauwert")
# plt.ylabel("Anzahl Pixel")
# plt.title("Histogramm der Grauwerte")
# plt.xlim(0, 256)
# plt.grid(axis="y", alpha=0.3)
# plt.show()